# Engineering roadmap: typed calculations through step 8

This executed workbench complements `process_to_engineering_simulator.ipynb`. It exercises the typed equipment-family, piping-network, valve, instrument, safety-scenario, materials, and preliminary mechanical calculations used by the eight-stage process-to-engineering workflow. Results are governed preliminary engineering outputs and remain review-required.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path(os.environ.get('NEQSIM_PROJECT_ROOT', Path.cwd())).resolve()
while not (ROOT / 'pom.xml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'devtools'))
from neqsim_dev_setup import neqsim_init, neqsim_classes
ns = neqsim_classes(neqsim_init(project_root=ROOT, recompile=not (ROOT / 'target/classes').exists(), verbose=False))
import jpype
import pandas as pd
JClass = ns.JClass
ArrayList = JClass('java.util.ArrayList')
LinkedHashMap = JClass('java.util.LinkedHashMap')
Context = JClass('neqsim.process.engineering.calculation.EngineeringCalculationContext')
context = Context.builder().designCaseId('maximum').build()
print('Loaded governed engineering calculations from', ROOT)

All NeqSim classes imported OK
Loaded governed engineering calculations from /workspace/scratch/0f75633c511c/neqsim


## Equipment-family calculations

Each result carries its design basis, method/version, inputs, governing case, preliminary uncertainty, constraints, and approval status.

In [2]:
Input = JClass('neqsim.process.engineering.calculation.EquipmentDesignCalculations$Input')
families = [
 ('separator', JClass('neqsim.process.engineering.calculation.EquipmentDesignCalculations$Separator')(), Input.builder('V-100', 'maximum', 'DB-001').value('gasFlowM3s', 2.0).value('liquidFlowM3s', 0.02).value('gasDensityKgM3', 35.0).value('liquidDensityKgM3', 750.0).build()),
 ('compressor', JClass('neqsim.process.engineering.calculation.EquipmentDesignCalculations$Compressor')(), Input.builder('K-100', 'maximum', 'DB-001').value('operatingFlowM3s', 5.0).value('surgeFlowM3s', 3.5).value('chokeFlowM3s', 7.0).value('shaftPowerKw', 2400.0).build()),
 ('pump', JClass('neqsim.process.engineering.calculation.EquipmentDesignCalculations$Pump')(), Input.builder('P-100', 'maximum', 'DB-001').value('flowM3s', 0.05).value('headM', 120.0).value('npshaM', 8.0).value('npshrM', 4.0).value('shaftPowerKw', 80.0).build()),
 ('heat exchanger', JClass('neqsim.process.engineering.calculation.EquipmentDesignCalculations$HeatExchanger')(), Input.builder('E-100', 'maximum', 'DB-001').value('dutyKw', 1000.0).value('overallU_WPerM2K', 500.0).value('correctedLmtdK', 25.0).build()),
 ('column', JClass('neqsim.process.engineering.calculation.EquipmentDesignCalculations$Column')(), Input.builder('T-100', 'maximum', 'DB-001').value('operatingVaporLoadM3s', 4.0).value('floodingVelocityMPerS', 2.0).value('minimumVaporLoadM3s', 1.5).build()),
 ('tank', JClass('neqsim.process.engineering.calculation.EquipmentDesignCalculations$Tank')(), Input.builder('TK-100', 'maximum', 'DB-001').value('normalInflowM3s', 0.02).value('workingTimeS', 600.0).value('emergencyTimeS', 300.0).build()),
]
equipment_rows = []
for family, module, inputs in families:
    result = module.calculate(inputs, context)
    values = result.getValue().toMap()
    equipment_rows.append({'family': family, 'status': str(result.getStatus()), 'method': str(module.getMethod()), 'governing case': str(values.get('governingCaseId'))})
equipment_table = pd.DataFrame(equipment_rows)
display(equipment_table)
assert len(equipment_table) == 6 and all(equipment_table.status == 'CALCULATED_REVIEW_REQUIRED')

,family,status,method,governing case
0,separator,CALCULATED_REVIEW_REQUIRED,separator-scrubber-preliminary-design,maximum
1,compressor,CALCULATED_REVIEW_REQUIRED,compressor-package-envelope-design,maximum
2,pump,CALCULATED_REVIEW_REQUIRED,pump-package-hydraulic-design,maximum
3,heat exchanger,CALCULATED_REVIEW_REQUIRED,heat-exchanger-preliminary-thermal-design,maximum
4,column,CALCULATED_REVIEW_REQUIRED,column-absorber-hydraulic-design,maximum
5,tank,CALCULATED_REVIEW_REQUIRED,tank-vessel-inventory-design,maximum


## Network piping and valve/instrument engineering

The network solver selects the smallest schedule candidate satisfying all declared cases and versioned project rules. Valve and instrument results expose phenomena and review decisions instead of silently approving them.

In [3]:
Piping = 'neqsim.process.engineering.piping.PipingNetworkDesignCalculation'
Candidate = JClass(Piping + '$Candidate'); HydraulicCase = JClass(Piping + '$Case'); Segment = JClass(Piping + '$Segment'); NetworkInput = JClass(Piping + '$Input')
candidates = ArrayList(); candidates.add(Candidate('4in', '40', 0.10, 100.0)); candidates.add(Candidate('8in', '40', 0.20, 100.0))
segment = Segment('L-100', True, False, 1000.0, 100.0, 10.0, 1.0, 0.0).addCase(HydraulicCase('maximum', 0.30, 0.10, 0.20, 50.0, 35.0))
segments = ArrayList(); segments.add(segment)
demands = LinkedHashMap(); demands.put('export', 0.30)
rules = JClass('neqsim.process.engineering.piping.PipingRulePack').norsokP0022023Ac2024()
network = JClass(Piping)().calculate(NetworkInput('export-network', rules, candidates, segments, demands), context)
selection = network.getValue().toMap().get('selections').get('L-100').get('candidate')
ValveInput = JClass('neqsim.process.engineering.calculation.ValveInstrumentDesignCalculations$ValveInput')
Failure = JClass('neqsim.process.engineering.calculation.ValveInstrumentDesignCalculations$FailurePosition')
valve = JClass('neqsim.process.engineering.calculation.ValveInstrumentDesignCalculations$Valve')().calculate(ValveInput('FV-100', 'maximum', 80.0, 100.0, 70.0, 50.0, 30.0, 10.0, 46.0, 600.0, 5.0, 5.0, 8.0, Failure.HAZOP_INPUT_REQUIRED), context)
InstrumentInput = JClass('neqsim.process.engineering.calculation.ValveInstrumentDesignCalculations$InstrumentInput')
instrument = JClass('neqsim.process.engineering.calculation.ValveInstrumentDesignCalculations$Instrument')().calculate(InstrumentInput('PIT-100', 'maximum', 20.0, 55.0, 0.0, 100.0, 0.005, 1.0, 0.5, 4.0, 10.0, 2.0, 30.0), context)
display(pd.DataFrame([{'calculation': 'network piping', 'status': str(network.getStatus()), 'selection': str(selection)}, {'calculation': 'valve', 'status': str(valve.getStatus()), 'selection': str(valve.getValue().toMap().get('failurePositionProposal'))}, {'calculation': 'instrument', 'status': str(instrument.getStatus()), 'selection': str(instrument.getValue().toMap().get('calibratedSpan'))}]))
assert str(selection.get('nominalSize')) == '8in'

,calculation,status,selection
0,network piping,CALCULATED_REVIEW_REQUIRED,"{nominalSize=8in, schedule=40, insideDiameterM..."
1,valve,CALCULATED_REVIEW_REQUIRED,HAZOP_INPUT_REQUIRED
2,instrument,CALCULATED_REVIEW_REQUIRED,100.0


## Governed safety scenarios, materials, and preliminary mechanics

Scenario credibility remains a controlled HAZOP input. Materials selection is a review-required degradation screen, and the mechanical result is preliminary—not a code or fabrication release.

In [4]:
Safety = 'neqsim.process.engineering.safety.SafetyScenarioEngineCalculation'
Scenario = JClass(Safety + '$Scenario'); ScenarioType = JClass(Safety + '$Type'); Credibility = JClass(Safety + '$Credibility'); Fluid = JClass(Safety + '$FluidModel')
scenario = Scenario('blocked-outlet', 'V-100', ScenarioType.BLOCKED_OUTLET, Credibility.CREDIBLE, Fluid.GAS, 'train-a', 'HAZOP-001', 10.0, 50.0, 0.20, 2.0, 5.0, 1.0, -20.0)
scenarios = ArrayList(); scenarios.add(scenario)
orifices = jpype.JArray(jpype.JDouble)([0.110, 0.196, 0.307])
safety_input = JClass(Safety + '$Input')('safety-study', scenarios, orifices, 3.0, 10.0, 120.0, -46.0)
safety = JClass(Safety)().calculate(safety_input, context)
MaterialInput = JClass('neqsim.process.engineering.calculation.MaterialsMechanicalDesignCalculations$MaterialInput')
material = JClass('neqsim.process.engineering.calculation.MaterialsMechanicalDesignCalculations$MaterialSelection')().calculate(MaterialInput('V-100', 'maximum', 0.02, 0.001, 2000.0, True, True, 40.0, 80.0, -46.0, 55.0, 3.0, 25.0, 'marine offshore external environment'), context)
MechanicalInput = JClass('neqsim.process.engineering.calculation.MaterialsMechanicalDesignCalculations$MechanicalInput')
mechanical = JClass('neqsim.process.engineering.calculation.MaterialsMechanicalDesignCalculations$PreliminaryMechanical')().calculate(MechanicalInput('V-100', 'maximum', 55.0, 80.0, 1.5, 4.5, 138.0, 0.85, 3.0, -46.0, 0.10), context)
review_rows = [
 {'calculation': 'relief/blowdown/flare scenarios', 'status': str(safety.getStatus()), 'result': str(safety.getValue().toMap().get('scenarios'))},
 {'calculation': 'materials', 'status': str(material.getStatus()), 'result': str(material.getValue().toMap().get('preliminaryMaterialClass'))},
 {'calculation': 'mechanical', 'status': str(mechanical.getStatus()), 'result': str(mechanical.getValue().toMap().get('preliminaryNominalThicknessMm'))},
]
display(pd.DataFrame(review_rows))
assert all(row['status'] == 'CALCULATED_REVIEW_REQUIRED' for row in review_rows)

,calculation,status,result
0,relief/blowdown/flare scenarios,CALCULATED_REVIEW_REQUIRED,"{blocked-outlet={scenarioId=blocked-outlet, pr..."
1,materials,CALCULATED_REVIEW_REQUIRED,SOUR_SERVICE_CARBON_STEEL_ISO_15156_SCREEN
2,mechanical,CALCULATED_REVIEW_REQUIRED,39.0


## How the notebooks cover the eight stages

This workbench proves the typed calculations in stages 4–6. `process_to_engineering_simulator.ipynb` proves stages 1–3 and 7–8: isolated iterative convergence, process-value and physical-variable stability, DEXPI 2.0 PFD/P&ID serialization, semantic round-trip evidence, coordinated registers/datasheets/reports, and the unresolved-action register. Final HAZOP/LOPA, SIL, vendor, metallurgy, code-design, layout, and construction approvals remain outside automatic authorization.